# ANÁLISIS EXPLORATORIO DE DATOS Y SISTEMA DE RECOMENDACIÓN HÍBRIDO SOBRE TENDENCIAS CINEMATOGRÁFICAS MUNDIALES (2010–2025)
## Documento Científico / Técnico - Fase 3: Preparación de Datos (Data Preparation)

---

### **INFORMACIÓN GENERAL Y METADATOS ACADÉMICOS**
- **Asignatura:** Programación para la Ciencia de Datos
- **Profesor:** Omar Salinas Silva
- **Fecha de Entrega:** Domingo 8 de Noviembre de 2026
- **Integrantes del Grupo 4:**
  1. Daniela Baeza Rocha
  2. Juan González Elgueta
  3. Elías Manríquez Armijo
  4. Yestxander Rada Gutierrez

---

## FASE 3: PREPARACIÓN DE LOS DATOS (DATA PREPARATION) - CRISP-DM

### 1. INTRODUCCIÓN Y METODOLOGÍA DEL RECOMENDADOR HÍBRIDO
En el marco de la metodología **CRISP-DM**, la fase de **Preparación de Datos** consolida el pipeline de ingeniería de características (*Data Engineering*) que transforma el conjunto de datos crudo (`movies_raw.csv`, compuesto por 17,978 producciones cinematográficas reales de TMDB entre 2010 y 2025) en un dataset analítico y matricial optimizado para alimentarse en un **Sistema de Recomendación Híbrido (Contenido + Calidad/Popularidad)**.

#### **Enfoque:**
El objetivo central de un recomendador híbrido es resolver dos preguntas fundamentales del usuario:
1. **Dimensión de Contenido (NLP & Vectorización):** *¿Qué películas se parecen temáticamente a la película de interés?* (Procesando `genres`, `production_companies` y `original_language`).
2. **Dimensión de Calidad y Tracción (IMDb Weighted Rating & Popularity):** *De todas las películas similares, ¿cuáles tienen credibilidad estadística comprobada y alta valoración de la audiencia?*

#### **Pipeline Secuencial de Preparación:**
- **Paso 1: Auditoría e Ingesta Inmutable:** Ingesta robusta y detección de vacíos estructurados.
- **Paso 2: Limpieza Textual y Formateo de Contenido:** Tratamiento de listas de géneros y productoras.
- **Paso 3: Justificación Metodológica contra el One-Hot Encoding:** Explicación técnica del descarte de OHE en favor de la matriz dispersa *CountVectorizer / Metadata Soup*.
- **Paso 4: Ingeniería de Métricas de Calidad (Weighted Rating de IMDb):** Construcción de la métrica $WR$ ponderada por representatividad muestral ($m = 	ext{percentil 75}$).
- **Paso 5: Normalización y Escalado Adaptativo:** Aplicación de `MinMaxScaler` para fusionar escalas heterogéneas ($[0, 1]$).
- **Paso 6: Construcción de la Matriz de Salida de Recomendación (`df_recommender`):** Consolidación matricial manteniendo identificadores explícitos (`movie_id`, `title`).

### 1.1 Carga del Entorno de Trabajo y Librerías Científicas
**Justificación Técnica:** Se cargan las herramientas fundamentales del ecosistema Python: `pandas` y `numpy` para la manipulación matricial, `matplotlib` y `seaborn` para diagnóstico visual, `pathlib.Path` para gestión de rutas, `sklearn.feature_extraction.text.CountVectorizer` para el procesamiento del lenguaje natural (NLP) y `MinMaxScaler` para la normalización acotada de puntuaciones.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import MinMaxScaler

# Configuración del entorno de visualización científica
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("Ecosistema de librerías para el Recomendador Híbrido inicializado correctamente.")

Ecosistema de librerías para el Recomendador Híbrido inicializado correctamente.


### 1.2 Adquisición e Ingesta del Dataset TMDB Original (17,978 Películas)
**Justificación Técnica:** Implementamos la rutina inmutable con `Path` para verificar la presencia local de `movies_raw.csv` en la ruta del proyecto (`../data/raw/` o directa) o activar la carga interactiva mediante `google.colab.files.upload()`.

In [2]:
np.random.seed(42)  # Semilla para reproducibilidad del pipeline

rutas_posibles = [
    Path("../data/raw/movies_raw.csv"),
    Path("movies_raw.csv")
]

ruta = next((p for p in rutas_posibles if p.exists()), None)

if ruta is None:
    try:
        from google.colab import files
        cargados = files.upload()
        nombre_archivo = next(iter(cargados))
        ruta = Path(nombre_archivo)
    except Exception as e:
        raise FileNotFoundError(
            "No se encontró movies_raw.csv. Súbelo al entorno de ejecución."
        ) from e

df_raw = pd.read_csv(ruta)
df_prep = df_raw.copy()

print(f"Archivo cargado: {ruta}")
print(f"Volumen Muestral: {df_prep.shape[0]:,} registros | {df_prep.shape[1]} columnas")
df_prep.head()

Archivo cargado: ..\data\raw\movies_raw.csv
Volumen Muestral: 17,978 registros | 16 columnas


,movie_id,title,release_date,status,budget,revenue,runtime,genres,production_companies,production_countries,spoken_languages,original_language,vote_average,vote_count,popularity,adult
0,113727,Dark Seduction,2010-01-01,Released,0,0,89,Drama|Thriller|Romance,Lionsgate,Mexico,Español,es,6.799,102,1.6261,False
1,67250,12,2010-01-01,Released,0,0,3,Comedy,NaN,United States of America,English,en,5.000,57,0.3124,False
2,43615,"Lula, the Son of Brazil",2010-01-01,Released,9500000,3785593,130,Drama|History,LC Barreto|Filmes do Equador|InterVideo Produc...,Brazil,Português,pt,5.500,90,1.3337,False
3,38883,Protect and Serve,2010-01-03,Released,0,3807808,90,Comedy|Crime|Action,M6 Films|Same Player|Pathé|SCOPE Pictures|Webe...,France,Français,fr,4.500,106,0.5622,False
4,62105,Re-Cut,2010-01-04,Released,0,0,80,Horror|Thriller,Fritz Manger Productions|National Picture Show,United States of America,NaN,en,6.535,115,0.7341,False


---

## PASO 1: AUDITORÍA DE METADATOS Y DIAGNÓSTICO DE COMPONENTES DEL RECOMENDADOR

### 1.1 Auditoría de Valores Faltantes en Columnas Clave
**Justificación Técnica:** A diferencia del análisis de regresión donde se buscaban nulos numéricos en `revenue`/`budget`, para un recomendador híbrido auditamos la existencia de ausencias en las variables temáticas (`genres`, `production_companies`, `original_language`) y cuantitativas de votación (`vote_average`, `vote_count`, `popularity`).

In [3]:
target_cols = ['movie_id', 'title', 'genres', 'production_companies', 'original_language', 'vote_average', 'vote_count', 'popularity']
audit_df = pd.DataFrame({
    'Tipo de Dato': df_prep[target_cols].dtypes,
    'Valores Nulos (NaN)': df_prep[target_cols].isnull().sum(),
    'Porcentaje Nulos (%)': np.round((df_prep[target_cols].isnull().sum() / len(df_prep)) * 100, 2)
})
audit_df

,Tipo de Dato,Valores Nulos (NaN),Porcentaje Nulos (%)
movie_id,int64,0,0.00
title,str,0,0.00
genres,str,10,0.06
production_companies,str,494,2.75
original_language,str,0,0.00
vote_average,float64,0,0.00
vote_count,int64,0,0.00
popularity,float64,0,0.00


**Interpretación Analítica e Impacto:**
- **`genres` (10 nulos, $0.06\%$):** Presenta una pérdida insignificante pero crítica para la similitud. Imputaremos con la cadena `'Unknown'` para evitar errores en la tokenización.
- **`production_companies` (494 nulos, $2.75\%$):** La ausencia de productora se imputará como `'Unknown'` para conservar la película dentro del catálogo consultable.
- **`vote_average`, `vote_count`, `popularity` ($0$ nulos):** Las métricas numéricas de recepción están completas al $100\%$, permitiendo el cálculo inmediato de los scores de credibilidad.

---

## PASO 2: LIMPIEZA TEXTUAL Y CONSTRUCCIÓN DE LA "SOPA DE METADATOS" (*METADATA SOUP*)

### 2.1 Normalización de Cadenas y Creación de Cadenas de Contenido Combinado
**Justificación Técnica:** En la columna `genres`, los elementos vienen separados por tuberías (ej. `Drama|Thriller|Romance`). Para procesarlos mediante modelos de espacio vectorial (NLP), debemos transformar estas cadenas separadas por caracteres especiales en texto continuo normalizado en minúsculas y sin espacios internos en nombres de productoras (ej. `WarnerBros` en lugar de `Warner Bros`) para evitar que el vectorizador confunda palabras individuales.

In [4]:
# 1. Imputación de nulos categóricos
df_prep['genres'] = df_prep['genres'].fillna('Unknown')
df_prep['production_companies'] = df_prep['production_companies'].fillna('Unknown')
df_prep['original_language'] = df_prep['original_language'].fillna('Unknown')

# 2. Función de limpieza y normalización textual
def clean_genres(text):
    return text.replace('|', ' ').lower()

def clean_text_feature(text):
    # Elimina espacios dentro de los nombres de productoras para tratarlas como entidades únicas
    tokens = text.split('|')
    cleaned_tokens = [t.replace(' ', '').lower() for t in tokens[:3]] # Tomamos las top 3 productoras
    return ' '.join(cleaned_tokens)

df_prep['genres_clean'] = df_prep['genres'].apply(clean_genres)
df_prep['companies_clean'] = df_prep['production_companies'].apply(clean_text_feature)
df_prep['lang_clean'] = df_prep['original_language'].str.lower()

# 3. Construcción de la Sopa de Metadatos (Metadata Soup)
df_prep['metadata_soup'] = df_prep['genres_clean'] + " " + df_prep['companies_clean'] + " " + df_prep['lang_clean']

print("Muestra de la 'Metadata Soup' generada para vectorización:")
df_prep[['title', 'metadata_soup']].head()

Muestra de la 'Metadata Soup' generada para vectorización:


,title,metadata_soup
0,Dark Seduction,drama thriller romance lionsgate es
1,12,comedy unknown en
2,"Lula, the Son of Brazil",drama history lcbarreto filmesdoequador interv...
3,Protect and Serve,comedy crime action m6films sameplayer pathé fr
4,Re-Cut,horror thriller fritzmangerproductions nationa...
